## Foursquare H3 Index Enrichment

Joins `checkins_raw` (90M rows) with `pois` on `venue_id` to attach coordinates,
then computes [H3 cell indexes](https://h3geo.org/) at four resolutions as **BIGINT** using
Databricks' built-in `h3_longlatash3` function, and derives a `local_time` column from
`utc_time` + `timezone_offset_minutes`. Table is liquid-clustered on `user_id`, `h3_r9`, `local_time`.

| Resolution | Avg. hexagon area | Approx. scale |
|---|---|---|
| 5 | ~252 km² | City / metro |
| 7 | ~5.2 km² | Neighborhood |
| 9 | ~0.1 km² | City block |
| 12 | ~0.0003 km² | Building |

Output table: **`{catalog}.{schema}.checkins_h3`** (configured via widgets above)

In [0]:
# utc_time strings use the legacy format 'EEE MMM dd HH:mm:ss Z yyyy' (e.g. 'Sun Jan 13 03:55:07 +0000 2013')
# which Spark 3.0+'s strict DateTimeFormatter can't parse; the legacy policy restores the Spark 2.x behaviour.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.{schema}.checkins_h3
  CLUSTER BY (user_id, h3_r9, local_time)  -- optimised for user-trajectory and block-level spatial queries
  AS
  SELECT
    c.user_id,
    c.venue_id,
    c.utc_time,
    c.timezone_offset_minutes,
    p.latitude,
    p.longitude,
    p.venue_category_name,
    p.country_code,
    -- H3 indexes as BIGINT at four resolutions
    h3_longlatash3(p.longitude, p.latitude,  3) AS h3_r3,
    h3_longlatash3(p.longitude, p.latitude,  5) AS h3_r5,
    h3_longlatash3(p.longitude, p.latitude,  7) AS h3_r7,
    h3_longlatash3(p.longitude, p.latitude,  9) AS h3_r9,
    h3_longlatash3(p.longitude, p.latitude, 12) AS h3_r12,
    -- Local time: parse UTC string then shift by the per-row timezone offset
    -- try_to_timestamp returns NULL for malformed rows (some utc_time fields contain embedded tweet text)
    TIMESTAMPADD(
      MINUTE,
      c.timezone_offset_minutes,
      try_to_timestamp(c.utc_time, 'EEE MMM dd HH:mm:ss Z yyyy')
    ) AS local_time
  FROM {catalog}.{schema}.checkins_raw  c
  JOIN {catalog}.{schema}.pois          p  ON c.venue_id = p.venue_id
  WHERE p.latitude  IS NOT NULL
    AND p.longitude IS NOT NULL
""")

row_count = spark.table(f"{catalog}.{schema}.checkins_h3").count()
print(f"checkins_h3 created: {row_count:,} rows")

checkins_h3 created: 90,048,627 rows


In [0]:
%sql
OPTIMIZE IDENTIFIER(:catalog || '.' || :schema || '.checkins_h3')

path,metrics
s3://justinm-demo-ext-s3-049629455384-27i4vf/__unitystorage/catalogs/0ebf4dde-03a2-48dc-94b9-41c907eebed9/tables/042f62cd-78dd-4fbe-805b-525810adc2fe,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 64, 0, false, 0, 0, 1782328807950, 1782328815285, 32, 0, null, List(0, 0), null, 14, 14, 0, 0, List(3794016952, false, false, false, 0.9841107133282266, List(0.7473464387052384, 0.7422897896331232, 0.7451934956582605), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(383, 99, 2413, 0, 0, 0), 2, 1, 5, sizeAware, false, 0, null, false, 0, 0, 0, null, null, null), null)"
s3://justinm-demo-ext-s3-049629455384-27i4vf/__unitystorage/catalogs/0ebf4dde-03a2-48dc-94b9-41c907eebed9/tables/042f62cd-78dd-4fbe-805b-525810adc2fe,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 64, 0, true, 0, 0, 1782328815333, 1782328818131, 32, 0, null, List(0, 0), null, 14, 14, 0, 0, List(3794016952, false, false, false, 0.9841107133282266, List(0.7473464387052384, 0.7422897896331232, 0.7451934956582605), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 669, 0, 0, 0), 15, 1, 1, null, false, 0, null, false, 0, 0, 0, null, null, null), null)"


In [0]:
# Sample rows
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print("=== Sample rows ===")
display(spark.table(f"{catalog}.{schema}.checkins_h3").limit(10))

=== Sample rows ===


user_id,venue_id,utc_time,timezone_offset_minutes,latitude,longitude,venue_category_name,country_code,h3_r3,h3_r5,h3_r7,h3_r9,h3_r12,local_time
1025890,4e6de6d814955284ba72714d,Mon Jan 28 14:20:26 +0000 2013,-180,-1.453803,-48.483575,College Gym,BR,592228092643115007,601235247874441215,610242446172880895,619249645411631103,632760444293560831,2013-01-28T11:20:26.000Z
1150693,50a9bd68e4b05dd8c3fe492d,Sat Jan 26 07:42:41 +0000 2013,420,-7.647179,111.324038,Asian Restaurant,ID,592460982949773311,601468161803419647,610475360202522623,619482559452282879,632993358334243327,2013-01-26T14:42:41.000Z
1229302,4ff82d13e4b0f1abbc018a74,Sat Jan 26 14:04:46 +0000 2013,420,-6.209904,106.825682,Residential Building (Apartment / Condo),ID,592435625462857727,601442789284118527,610449987649667071,619457186902048767,632967985784081407,2013-01-26T21:04:46.000Z
1082080,4ba5f5abf964a520612b39e3,Mon Jan 28 13:33:04 +0000 2013,420,-6.156854,106.893282,Church,ID,592435625462857727,601442942829199359,610450141765173247,619457341012049919,632968139894073343,2013-01-28T20:33:04.000Z
1285646,4c7fdeaa6cc2b71390d50c9b,Wed Apr 17 12:23:17 +0000 2013,-240,-3.134144,-60.012512,University,BR,592416246570418175,601423415760388095,610430614511812607,619437813755281407,632948612637150719,2013-04-17T08:23:17.000Z
1040947,4d2072da5acaa35dc0e6c735,Sun Apr 07 12:14:31 +0000 2013,-180,-25.409545,-49.216012,Church,BR,592930474414833663,601937619982483455,610944818213814271,619952017463050239,633462816345102847,2013-04-07T09:14:31.000Z
1463513,4e7c8f02cc21f4555f65c4c9,Mon Apr 08 13:13:28 +0000 2013,420,-7.938554,112.629278,Soccer Stadium,ID,592461257827680255,601468440976293887,610475639744495615,619482838985342975,632993637867262975,2013-04-08T20:13:28.000Z
1361402,4b981fcff964a520202e35e3,Wed Apr 10 16:43:13 +0000 2013,-240,-33.441657,-70.654074,Plaza,CL,593116566757834751,602123720915419135,611130919129972735,620138118368985087,633648917251005951,2013-04-10T12:43:13.000Z
1125346,4ba689ccf964a5204d5c39e3,Thu Apr 04 22:14:32 +0000 2013,-180,-26.912126,-49.099577,Bakery,BR,592930749292740607,601937916335226879,610945115036319743,619952314283196415,633463113168153087,2013-04-04T19:14:32.000Z
1070572,4fc6d6aee4b0b4c8bc4ff0a7,Tue Feb 26 10:24:44 +0000 2013,420,-7.779366,110.415465,Trade School,ID,592461738864017407,601468919865147391,610476118918561791,619483318167273471,632994117049280511,2013-02-26T17:24:44.000Z


In [0]:
%sql
SELECT
  COUNT(DISTINCT h3_r3)  AS distinct_r3_cells,
  COUNT(DISTINCT h3_r5)  AS distinct_r5_cells,
  COUNT(DISTINCT h3_r7)  AS distinct_r7_cells,
  COUNT(DISTINCT h3_r9)  AS distinct_r9_cells,
  COUNT(DISTINCT h3_r12) AS distinct_r12_cells,
  MIN(local_time)        AS earliest_local_time,
  MAX(local_time)        AS latest_local_time
FROM IDENTIFIER(:catalog || '.' || :schema || '.checkins_h3')

distinct_r3_cells,distinct_r5_cells,distinct_r7_cells,distinct_r9_cells,distinct_r12_cells,earliest_local_time,latest_local_time
7560,87274,571617,2663430,8959566,2012-04-03T08:03:13.000Z,2014-01-30T03:32:48.000Z
